# Week 4 — Visualizing Handwritten Digits with PCA

**Theme:** Unsupervised learning I — dimensionality reduction (PCA)

Unlike last week, this week's algorithm gets **no labels at all**. It never
sees "this is a 3" — it only sees 64 pixel-brightness numbers per image and
looks for the directions in which those numbers vary the most. That's
**Principal Component Analysis (PCA)**: a way to compress high-dimensional data
into a few numbers while keeping as much of the original variation as possible.

**Dataset:** scikit-learn's built-in `digits` dataset — 1,797 handwritten digit
images, each an 8x8 grid of pixel brightness values (0-16), i.e. 64 numbers per image.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target  # X: (1797, 64), y: the true digit 0-9 (only used for coloring plots, not for PCA itself)
print("Shape of X:", X.shape)

In [ ]:
# Look at a few raw digit images
fig, axes = plt.subplots(2, 5, figsize=(8, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap="gray_r")
    ax.set_title(str(digits.target[i]))
    ax.axis("off")
plt.suptitle("A few raw 8x8 digit images (64 pixels each)")
plt.show()

## Compress 64 numbers down to 2

PCA finds the 2 directions (out of 64 possible) along which the digit images
vary the most, and projects every image onto just those 2 numbers.

In [ ]:
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)
print("Compressed shape:", X_2d.shape)
print(f"These 2 components explain {pca.explained_variance_ratio_.sum():.1%} of the total variation.")

In [ ]:
plt.figure(figsize=(7, 6))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap="tab10", s=15, alpha=0.7)
plt.colorbar(scatter, label="Digit")
plt.title("1,797 Handwritten Digits Compressed to 2D with PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

Notice: PCA was never told which image was a 0, 3, or 7 — but images of the
same digit still land near each other in the compressed 2D space, just because
they tend to look similar (similar pixel patterns).

## How many components do we actually need?

We used 2 components just so we could plot them. How much information would we
keep with more?

In [ ]:
pca_full = PCA().fit(X)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker=".")
plt.axhline(0.95, color="red", linestyle="--", label="95% of variance")
plt.title("Cumulative Explained Variance vs. Number of Components")
plt.xlabel("Number of components")
plt.ylabel("Cumulative variance explained")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

n_for_95 = int(np.argmax(cumulative_variance >= 0.95) + 1)
print(f"Components needed for 95% of the variance: {n_for_95} (out of 64 original pixels)")

## What do we lose by compressing? Reconstruct the images

PCA compression is lossy — let's compress each image down to just 2 numbers,
then reconstruct it back to 64 pixels, and see how blurry it gets.

In [ ]:
pca_2 = PCA(n_components=2).fit(X)
X_reconstructed_2 = pca_2.inverse_transform(pca_2.transform(X))

pca_20 = PCA(n_components=20).fit(X)
X_reconstructed_20 = pca_20.inverse_transform(pca_20.transform(X))

fig, axes = plt.subplots(3, 6, figsize=(10, 5))
for i in range(6):
    axes[0, i].imshow(X[i].reshape(8, 8), cmap="gray_r")
    axes[1, i].imshow(X_reconstructed_2[i].reshape(8, 8), cmap="gray_r")
    axes[2, i].imshow(X_reconstructed_20[i].reshape(8, 8), cmap="gray_r")
    for row in range(3):
        axes[row, i].axis("off")
axes[0, 0].set_title("orig", loc="left")
axes[0, 0].text(-4, 4, "original\n(64 numbers)", fontsize=8)
axes[1, 0].text(-4, 4, "2 components", fontsize=8)
axes[2, 0].text(-4, 4, "20 components", fontsize=8)
plt.suptitle("Original vs. PCA-Reconstructed Digits")
plt.tight_layout()
plt.show()

## Try it yourself

1. **3D instead of 2D.** Set `n_components=3` and make a 3D scatter plot
   (`fig.add_subplot(projection="3d")`) — do the digit clusters separate more cleanly?
2. **Which digits get confused?** Looking at the 2D scatter plot, which two
   digits' clusters overlap the most? Does that match digits that look similar
   when handwritten (e.g. 4 and 9, or 3 and 8)?
3. **Reconstruction quality.** Try reconstructing with `n_components=1` — how
   bad does it look, and why?
4. **Connect it to Week 1.** Compute `np.corrcoef` between PC1 and PC2 across
   all digits — should it be close to 0? (Hint: PCA components are, by
   construction, uncorrelated with each other.)